# Control loop from a notebook

Talks to an Arduino running `ControlDemo` over [CtrlLink](../PROTOCOL.md): set gains,
step the reference, get the timeseries back as a DataFrame.

`sync_board()` below builds, flashes and connects, doing only the parts that are
actually out of date.

**If a motor is connected**, the open-loop cells below will spin it. Check the bench
before running them.

In [165]:
import sys
sys.path.insert(0, '../python')

import hashlib
import json
import subprocess
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from ctrllink import CtrlLink, CtrlLinkError, find_port

plt.rcParams['figure.figsize'] = (9, 3.5)
plt.rcParams['axes.grid'] = True

## Building, flashing and connecting

`sync_board()` does the whole start-of-session dance in one call:

- **compiles** only if a source file actually changed (hashed, not timestamped, so a
  `git checkout` does not trigger a pointless rebuild),
- **uploads** only if the resulting binary differs from what this port was last given,
- **reopens the link**, which resets the board — so the loop starts from a known state
  whether or not anything needed flashing.

`force_compile=True` and `force_upload=True` override the two checks independently.
Call it again any time you edit the sketch; a no-op run takes about a second.

In [166]:
FQBN      = 'arduino:avr:uno'
SKETCH    = Path('../ControlDemo')
LIBRARIES = Path('../libraries')
BUILD_DIR = Path('../build')

_link = None


def _sources_hash():
    """Fingerprint of everything the sketch is built from.

    Contents rather than timestamps: a git checkout rewrites mtimes without
    changing a line, and would otherwise trigger a pointless rebuild.
    """
    digest = hashlib.sha256()
    files = sorted(list(SKETCH.glob('*.ino')) +
                   [p for p in LIBRARIES.rglob('*') if p.suffix in ('.h', '.cpp', '.c')])
    for path in files:
        digest.update(path.name.encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def _run(argv, what):
    done = subprocess.run(argv, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f'{what} failed:\n{(done.stdout + done.stderr).strip()}')
    return done.stdout + done.stderr


def _load_state():
    try:
        return json.loads((BUILD_DIR / 'sync-state.json').read_text())
    except (OSError, ValueError):
        return {}


def _save_state(state):
    BUILD_DIR.mkdir(parents=True, exist_ok=True)
    (BUILD_DIR / 'sync-state.json').write_text(json.dumps(state, indent=1))


def wait_for_port(hint=None, timeout=2.0):
    """find_port(), but tolerant of a board that is still re-enumerating.

    The default is short because an absent board should be reported at once;
    the long wait is only worth it just after an upload, when the bridge may
    genuinely take a few seconds to come back.
    """
    deadline = time.monotonic() + timeout
    while True:
        try:
            return find_port(hint)
        except CtrlLinkError:
            if time.monotonic() >= deadline:
                raise
            time.sleep(0.3)


def sync_board(port=None, force_compile=False, force_upload=False,
               connect=True, verbose=True):
    """Bring the board and the serial link up to date, and reconnect.

    Compiles only when a source file has actually changed, uploads only when the
    resulting binary differs from what this port was last given, and always
    reopens the link -- which resets the board, so the loop starts from a known
    state whether or not anything needed flashing.

    force_compile and force_upload each override their own check.
    """
    global _link

    def say(message):
        if verbose:
            print(message)

    state = _load_state()
    hex_file = BUILD_DIR / f'{SKETCH.name}.ino.hex'
    sources = _sources_hash()

    if force_compile or not hex_file.exists() or state.get('sources') != sources:
        say('compiling ...')
        output = _run(['arduino-cli', 'compile', '--fqbn', FQBN,
                       '--libraries', str(LIBRARIES.resolve()),
                       '--build-path', str(BUILD_DIR.resolve()),
                       str(SKETCH.resolve())], 'compile')
        for line in output.splitlines():
            if 'bytes' in line:
                say('  ' + line.strip())
        state['sources'] = sources
        _save_state(state)
    else:
        say('sources unchanged, not recompiling')

    binary = hashlib.sha256(hex_file.read_bytes()).hexdigest()

    if port is None:
        try:
            port = wait_for_port()
        except CtrlLinkError:
            raise CtrlLinkError(
                'the sketch is built, but no board is reachable: no USB serial '
                'port was found. Plug it in and run this again -- the build is '
                'cached, so it will go straight to uploading.') from None

    uploaded = state.get('uploaded', {})

    if force_upload or uploaded.get(port) != binary:
        # Uploading needs the port to itself, and resets the board regardless.
        if _link is not None:
            _link.close()
            _link = None
        say(f'uploading to {port} ...')
        _run(['arduino-cli', 'upload', '--fqbn', FQBN, '-p', port,
              '--input-dir', str(BUILD_DIR.resolve()),
              str(SKETCH.resolve())], 'upload')
        uploaded[port] = binary
        state['uploaded'] = uploaded
        _save_state(state)
        port = wait_for_port(timeout=15.0)  # some bridges drop off the bus while resetting
    else:
        say(f'{port} already has this binary, not uploading')

    if _link is not None:
        _link.close()
        _link = None

    if not connect:
        return None

    _link = CtrlLink(port)
    say(f'connected on {port}: {_link.info}')
    return _link

In [213]:
dev = sync_board()

compiling ...
  Sketch uses 13808 bytes (42%) of program storage space. Maximum is 32256 bytes.
  Global variables use 435 bytes (21%) of dynamic memory, leaving 1613 bytes for local variables. Maximum is 2048 bytes.
uploading to /dev/cu.usbserial-1110 ...
connected on /dev/cu.usbserial-1110: CtrlLink 1 ControlDemo chans=4 row=33 dt_us=1000


## What the board says it has

Nothing here is hard-coded on the Python side. The device is asked what it exposes,
so a gain added to the sketch shows up below with no change to this notebook.

`ref` and `refrate` carry 8 fractional bits and are in the raw units of whatever
`target` selects — counts for position, ADC LSBs for current. The two helpers below
turn engineering units into that representation using the scales the board reports,
so neither the counts per revolution nor the current-sense calibration is repeated
here.

In [ ]:
print('parameters:')
for name, value in dev.params.items():
    print(f'  {name:<9} {value}')

print('\ntelemetry channels:')
for c in dev.channels:
    print(f'  {c.name:<5} {c.type:<4} scale={c.scale:<12g} {c.unit}')

SCALE   = {c.name: c.scale for c in dev.channels}
REF_ONE = 1 / SCALE['ref']      # ref and refrate carry 8 fractional bits

# `mode` picks the controller, `target` picks the feedback it closes on.
MODE_OPEN, MODE_PID, MODE_RAMP = 0, 1, 2
POSITION, CURRENT = 0, 1

def ref_deg(deg):
    """Setpoint in degrees -> the integer `ref` wants, for target = POSITION."""
    return round(deg / SCALE['y_uw'] * REF_ONE)

def ref_ma(ma):
    """Setpoint in milliamps -> the integer `ref` wants, for target = CURRENT."""
    return round(ma / SCALE['i'] * REF_ONE)

def zero_here():
    """Call the shaft's present position zero.

    `y` reads as `offset - counts` wrapped, so moving `offset` down by the
    present `y` puts `offset` on the present count and `y` on zero. The
    unwrapped counter is zeroed after, once the loop has taken a sample with
    the new offset.
    """
    dev.offset = (dev.offset - dev.y) % 4096
    dev.y_uw   = 0


Parameters are plain attributes.

In [217]:
dev.kp = 2.5
print('kp is now', dev.kp)

# The value is read back and checked, so a mangled command cannot pass silently.
dev.kp = 0.5

kp is now 2.5


## Is the link healthy?

`capture()` checks this for you. It zeroes the device's health counters before the
run and reads them after, so `df.attrs` describes that capture and nothing else, and
it prints a note for anything that went wrong — a capture that quietly lost control
periods looks exactly like one that did not, until you go and ask.

What it watches:

- `missed` — control periods the loop never serviced. The sampler came round again
  before `loop()` had picked up the previous tick, so those periods did not run at
  all. This is the one that matters: the controller has a hole in it.
- `maxlate` — worst delay between a tick firing and the loop servicing it. Reported
  once it passes half the period, which is the loop keeping up but not by much.
- `drops` — rows the device could not fit into its transmit buffer.
- `gaps` — breaks in the tick sequence from any cause, including bytes lost in transit.
- `sovr` / `serr` — I2C transfers that missed their sample slot, or failed outright.

`dev.health()` reads the same counters directly, and `warn=False` on any capture
turns the printing off without losing the numbers.

In [ ]:
df = dev.capture(1.0)
span = df['t'].iloc[-1] - df['t'].iloc[0]

print(f'{len(df)} rows over {span:.3f} s  ->  {len(df)/span:.1f} Hz')
print(f'device sent {df.attrs["rows"]}, dropped {df.attrs["drops"]}, gaps {df.attrs["gaps"]}')
print(f'control period {df.attrs["dt_us"]} us, worst service delay {df.attrs["maxlate"]} us')
print(f'missed {df.attrs["missed"]}, sensor overruns {df.attrs["sovr"]}, '
      f'sensor errors {df.attrs["serr"]}')

if not df.attrs['health']:
    print('\nnothing to report')

df.head()


In [219]:
row_bytes = 4 + sum(c.width for c in dev.channels) + 1
rate = 1e6 / df.attrs['dt_us']
used = row_bytes * rate / (1_000_000 / 10)

print(f'{row_bytes} bytes/row x {rate:.0f} Hz = {row_bytes*rate/1000:.1f} kB/s')
print(f'{used*100:.0f}% of the 1 Mbaud link')

33 bytes/row x 1000 Hz = 33.0 kB/s
33% of the 1 Mbaud link


## The sensor

`y_uw` is the AS5600 angle, unwrapped: it accumulates through the 4096-count wrap
instead of jumping, so a shaft that keeps turning gives a monotonic trace. Turn the
magnet by hand while this runs and it should follow; a flat line means the magnet is
not moving (or is not there).

`y_uwf` is the same signal through the two-pole filter set by `tau_y`. With
`tau_y = 0` the filter is a pass-through and the two traces sit on top of each other,
which is the default: 50 ms of lag is a lot of phase to give away on a 1 kHz loop.
Whichever one you can see here is the one the position loop closes on.

In [ ]:
zero_here()
dev.tau_y = 0.05            # 50 ms, two poles

df = dev.capture(2.0)

plt.plot(df['t'], df['y_uw'],  label='y_uw  (raw)',  lw=0.8)
plt.plot(df['t'], df['y_uwf'], label='y_uwf (tau_y)')
plt.xlabel('t [s]'); plt.ylabel('angle [deg]'); plt.title('AS5600'); plt.legend()
plt.show()

print(f'raw      {df["y_uw"].min():.2f} .. {df["y_uw"].max():.2f} deg,  sd {df["y_uw"].std():.4f} deg')
print(f'filtered {df["y_uwf"].min():.2f} .. {df["y_uwf"].max():.2f} deg,  sd {df["y_uwf"].std():.4f} deg')

dev.tau_y = 0.0             # back to unfiltered feedback


## Open-loop step: identifying the plant

`mode = 0` bypasses the controller and puts `uff` straight on the actuator. Step it
and watch how the plant answers — that response is what the controller has to be
designed against.

`step()` holds for `pre` seconds, changes the parameter, then holds for `post`. The
board reports the exact tick the change landed on, so `t = 0` is the step itself, to
the sample — the host's timing jitter never enters the measurement.

In [ ]:
dev = sync_board()

dev.mode = MODE_OPEN
dev.uff  = 255              # spin up, then release on the step

df = dev.step('uff', 0, pre=1.0, post=1.0, back=0)

fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6))
a.plot(df['t'][1:], np.diff(df['y_uw']) / df.attrs['dt_us'] * 1e6 / 360)
a.set_ylabel('speed [rev/s]')
b.plot(df['t'], df['u']); b.set_ylabel('u [pwm]')
c.plot(df['t'], df['i']); c.set_ylabel('i [mA]'); c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('open-loop release')
plt.show()

print('step recorded at tick', df.attrs['marks'])
print(f'{(df["t"] < 0).sum()} samples before, {(df["t"] >= 0).sum()} after')


## Following a ramp

`mode = 2` adds `refrate` to `ref` every control period, so the setpoint sweeps at a
constant rate and the loop is asked to track a velocity rather than hold a point.
`refrate` is in the same fixed-point units as `ref`, which is why it can express less
than one count per period.

`target = POSITION` closes the loop on `y_uw`.

In [ ]:
dev = sync_board()

dev.mode   = MODE_OPEN
dev.uff    = 0
dev.target = POSITION

dev.kp, dev.ki, dev.kd = 0.001, 0.00005, 0.0

zero_here()
dev.ref = 0

# One revolution per second, as degrees of setpoint per control period.
rev_per_s   = 1.0
dev.refrate = ref_deg(rev_per_s * 360 * dev.tickdiv / 5000)
dev.mode    = MODE_RAMP

df = dev.step('refrate', 2 * dev.refrate, pre=5.0, post=5.0)

dev.mode = MODE_OPEN
dev.uff  = 0

# ref and e come back in target units, which for POSITION are counts.
ref_deg_ = df['ref'] * SCALE['y_uw']
e_deg    = df['e']   * SCALE['y_uw']
per_s    = 1e6 / df.attrs['dt_us'] / 360        # deg per sample -> rev/s

fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6))
a.plot(df['t'][1:], np.diff(ref_deg_)     * per_s, ls='--', label='ref')
a.plot(df['t'][1:], np.diff(df['y_uw'])   * per_s, label='y_uw')
a.set_ylabel('speed [rev/s]'); a.legend()
b.plot(df['t'], 100 * df['u'] / 255); b.set_ylabel('u [%]')
c.plot(df['t'], e_deg); c.set_ylabel('e [deg]'); c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('ramp: doubling the commanded speed')
plt.show()


## Closing the loop on current instead

`target = CURRENT` swaps the feedback: the same PID now works on `ref - i`, with `ref`
in ADC LSBs and `i` the filtered current sense. Nothing else about the loop changes —
the dispatch is a single branch in `control_error()` — so the gains, the ramp and the
anti-windup all behave the same way, only against a much faster plant.

Current needs far more filtering than position does; `tau_i` sets the two-pole corner
on the sense channel and `tau_e` the one-pole filter that feeds the derivative term.

In [ ]:
dev = sync_board()

dev.mode   = MODE_OPEN
dev.uff    = 0
dev.target = CURRENT

dev.tau_i = 0.005
dev.tau_e = 0.010
dev.kp, dev.ki, dev.kd = 0.1, 0.00005, 0.0

dev.ref  = ref_ma(120)
dev.mode = MODE_PID

df = dev.step('ref', ref_ma(140), pre=2.0, post=2.0, back=ref_ma(120))

dev.mode = MODE_OPEN
dev.uff  = 0

# target units are ADC LSBs here, so `i`'s own scale converts them to mA.
fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6))
a.plot(df['t'], df['ref'] * SCALE['i'], ls='--', label='ref')
a.plot(df['t'], df['i'], label='i')
a.set_ylabel('current [mA]'); a.legend()
b.plot(df['t'], 100 * df['u'] / 255); b.set_ylabel('u [%]')
c.plot(df['t'], df['e'] * SCALE['i']); c.set_ylabel('e [mA]'); c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color='k', lw=0.8, ls='--')
a.set_title('current step, 120 -> 140 mA')
plt.show()

print(f'missed {df.attrs["missed"]}, worst service delay {df.attrs["maxlate"]} us')


## Sweeping a gain

The whole point of driving this from a notebook: change a gain, re-measure, overlay.
Each pass is one round trip to the board.

In [ ]:
results = {}

dev.target = POSITION
dev.ki, dev.kd = 0.0, 0.0
zero_here()
dev.ref  = 0
dev.mode = MODE_PID

for kp in (0.5, 1.0, 2.0, 4.0):
    dev.kp = kp
    results[kp] = dev.step('ref', ref_deg(45), pre=0.05, post=0.45, back=0)

for kp, d in results.items():
    plt.plot(d['t'], d['y_uw'], label=f'kp = {kp}')
plt.plot(d['t'], d['ref'] * SCALE['y_uw'], 'k--', lw=0.8, label='ref')
plt.axvline(0, color='k', lw=0.8, ls='--')
plt.xlabel('t [s]'); plt.ylabel('y_uw [deg]'); plt.legend(); plt.title('kp sweep')
plt.show()

for kp, d in results.items():
    print(f'kp={kp:<5} rows={len(d):<5} drops={d.attrs["drops"]} '
          f'gaps={d.attrs["gaps"]} missed={d.attrs["missed"]}')

dev.mode = MODE_OPEN
dev.uff  = 0


## Changing the loop rate

`tickdiv` divides the 5 kHz sampler down to the control rate, so `tickdiv = 5` is 1 kHz
and `tickdiv = 50` is 100 Hz. Useful for showing what discretisation does to a loop that
was tuned at a different rate.

`dec` is separate: it thins the *telemetry* without touching the control rate, for when a
long run would otherwise outrun the link.

In [ ]:
dev.tickdiv = 25            # 5000 / 25 = 200 Hz
d = dev.capture(0.5)
print(f'tickdiv=25 -> period {d.attrs["dt_us"]} us, {len(d)/(d["t"].iloc[-1]-d["t"].iloc[0]):.0f} Hz')

dev.tickdiv = 5             # back to 1 kHz
d = dev.capture(0.5)
print(f'tickdiv=5  -> period {d.attrs["dt_us"]} us, {len(d)/(d["t"].iloc[-1]-d["t"].iloc[0]):.0f} Hz')

## Finishing up

Leave the actuator at rest and free the port, or the next `CtrlLink()` will find it busy.

In [ ]:
dev.mode = MODE_OPEN
dev.uff = 0
dev.close()
print('closed')